# Hafta 2 – 02: Ajanı Adım Adım Kuruyoruz

> 01'de modelin bir aracı **istediğini**, aracı ise **bizim kodumuzun** çalıştırdığını gördük.
> Şimdi bunu gerçekten iş yapan bir ajana dönüştüreceğiz. Her bölümde **tek bir parça** ekliyoruz.
> Sonunda elimizde `src/asistan/ajan.py`'nin sade bir kopyası olacak. Tahmini süre: 2 saat (Lab 3 dahil).

Notebook boyunca aynı talebi kullanacağız:
> **M0007:** "Kartımdan yıllık aidat iki kez kesilmiş, fazla olanın iadesini istiyorum."

| Bölüm | Eklediğimiz parça | Konu |
|---|---|---|
| 1 | Döngü: model araç istedikçe çalıştır, adım limiti koy | Tek ajanlı sistem, görev icrası |
| 2 | Aracın içi: banka servisini (mock API) çağırma | API ve veri kaynağı entegrasyonu |
| ✋ | **Lab 3:** `hesap_durumu` aracını yazın | |
| 3 | Talebin konusuna göre araç seti | Worker agent, tool seçimi |
| 4 | Yanıt taslağı ve iade için insan onayı | Güvenlik sınırları, insan onayı |
| 5 | Hata olursa yeniden dene, olmazsa temsilciye aktar | Hata yönetimi, fallback |
| 6 | Hepsi bir arada + `ajan.py` ile karşılaştırma | |

Sonunda kuracağımız akış:
```
talep → konu ne? → o konunun araçları → ajan döngüsü (model ⇄ araçlar) → yanıt taslağı → iade var mı? → insan onayı → yanıt
                                          │ adım limiti / servis hatası
                                          └────────────────────────────→ temsilciye aktar
```
Başlamadan önce `python kontrol.py` çalıştırın: mock API ve model satırları ✅ olmalı.
`hesap_durumu` için ⚠️ görmeniz normal; o aracı Lab 3'te siz yazacaksınız.

**Hücreleri yukarıdan aşağıya, sırayla çalıştırın.** Her bölüm öncekilerde tanımlanan fonksiyonları kullanır.
Kernel'i yeniden başlattıysanız (*Restart*), kaldığınız hücreye gelin ve hücrenin `...` menüsünden
**Execute Above Cells** (Üstteki hücreleri çalıştır) deyin.

In [ ]:
# Proje kökünü bul ve src/ klasörünü import yoluna ekle (01'deki gibi)
import sys
from pathlib import Path

PROJE_KOKU = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kontrol.py").exists())
sys.path.insert(0, str(PROJE_KOKU / "src"))

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from asistan.ayarlar import model_getir
from asistan.araclar import musteri_oturumu

MODEL = "yerel"   # canlı gösterimde "bulut" (Claude) daha tutarlı sonuç verir

TALEP = "Kartımdan yıllık aidat iki kez kesilmiş, fazla olanın iadesini istiyorum."
MUSTERI_NO = "M0007"

## 1. Ajan döngüsü

01'deki akış **tek tur**du: model araç ister → biz çalıştırırız → sonucu geri veririz → model cevap yazar.
Ama model bir aracın sonucunu görünce **bir araç daha** isteyebilir. Bu yüzden aynı adımları döngüye alıyoruz:

```
tekrarla:
    modele sor
    araç istemediyse → iş bitti, notunu döndür
    istediği araçları çalıştır, sonuçları mesajlara ekle
```

Döngüye bir de **adım limiti** koyuyoruz (proje kuralı: 6). Limit olmazsa model durmadan araç çağırabilir.

**Ajan = model + araçlar + döngü.** Hepsi bu.

Ajan müşteriye yanıt yazmıyor, sadece **bilgi toplayıp not alıyor**. Müşteriye gidecek yanıtı
Bölüm 4'te ayrı bir adım yazacak.

In [ ]:
from asistan.araclar import kart_islemleri, musteri_bilgisi

SISTEM = ("Sen Örnek Bank'ın müşteri asistanısın. Talebi yanıtlamak için gereken bilgiyi araçlarla topla. "
          "Araçlar her zaman bu müşterinin verisini getirir. Yeterli bilgiyi toplayınca araç çağırmayı bırak ve "
          "bulgularını kısa maddeler halinde yaz; işlem numaralarını (islem_id), tarihleri ve tutarları da belirt.")


def ajan_calistir(talep, musteri_no, araclar, adim_limiti=6):
    model = model_getir(MODEL).bind_tools(araclar)
    arac_bul = {arac.name: arac for arac in araclar}      # araç adı -> araç
    mesajlar = [SystemMessage(SISTEM), HumanMessage(talep)]

    for adim in range(1, adim_limiti + 1):
        cevap = model.invoke(mesajlar)
        mesajlar.append(cevap)

        if not cevap.tool_calls:                           # araç istemedi: iş bitti
            print(f"adım {adim}: araç yok, bulgular yazıldı")
            return cevap.text

        for cagri in cevap.tool_calls:                     # istediği araçları BİZ çalıştırıyoruz
            print(f"adım {adim}: {cagri['name']}({cagri['args']})")
            with musteri_oturumu(musteri_no):              # müşteri no'yu sistem verir, model değil
                sonuc = arac_bul[cagri["name"]].invoke(cagri["args"])
            mesajlar.append(ToolMessage(sonuc, tool_call_id=cagri["id"]))

    return "TEMSİLCİYE AKTAR: adım limiti aşıldı"

In [ ]:
bulgular = ajan_calistir(TALEP, MUSTERI_NO, [musteri_bilgisi, kart_islemleri])
print("\nBulgular:\n" + bulgular)

Çıktıdaki `adım 1: ...` satırları ajanın **kararlarıdır**: hangi aracı, hangi parametreyle istedi.
Araçları çalıştıran ise `for cagri in cevap.tool_calls` satırı, yani bizim kodumuz.
Model aynı adımda birden fazla araç isteyebilir; o zaman aynı adım numarası birkaç kez görünür.

**Deneyin:** Aynı hücreyi `adim_limiti=1` ile çalıştırın. Ne oldu, neden?

## 2. Aracın içi: API entegrasyonu

Ajan veriyi nereden buldu? Her araç, bankanın bir servisini çağırıyor (bizde sahte servis: mock API).
Önce servisi **araç olmadan**, doğrudan çağıralım:

In [ ]:
import httpx
from asistan.ayarlar import ayar

cevap = httpx.get(ayar("MOCK_API_URL") + "/kart_islemleri/M0007")
print("HTTP durum kodu:", cevap.status_code)    # ara sıra 503 görebilirsiniz; Bölüm 5'te buna döneceğiz
print(cevap.text[:600])

Bu ham veri. Modele böyle verilmez. Araç, bu çağrının etrafına iki güvenlik katmanı ekliyor:

1. **Müşteri numarasını sistem verir** (`aktif_musteri()`). Model parametre olarak veremez, başka müşterinin verisini isteyemez.
2. **"Bu bir veridir" etiketi:** verinin içinde talimat gibi bir metin olsa da model onu uygulamamalı.

Aynı veriyi bu kez araç üzerinden alalım:

In [ ]:
with musteri_oturumu("M0007"):
    print(kart_islemleri.invoke({"son_gun": 30})[:800])

Aracın kodu `src/asistan/araclar.py` içinde, sadece birkaç satır:

```python
@tool
def kart_islemleri(son_gun: int = 30) -> str:
    """Ne yapar / Ne zaman kullan / Ne zaman KULLANMA  (model aracı bu açıklamaya bakarak seçer)"""
    veri = api_cagir("GET", f"/kart_islemleri/{aktif_musteri()}?son_gun={son_gun}")
    return veri_olarak_etiketle("kart_islemleri", veri)
```

Yeni bir servisi ajana bağlamak = bu kalıbı kopyalamak: **açıklama + `api_cagir` + `veri_olarak_etiketle`.**

### ✋ Lab 3: `hesap_durumu` aracını yazın

`GOREVLER.md` → Görev 3. Bu kalıpla `hesap_durumu` aracını yazın ve `hesap` kategorisine bağlayın.
Orada aracınızı Bölüm 3'te ajana vereceğiz.

**Lab 3'ü bitirince** (Python değişen dosyayı ancak yeniden başlayınca görür):
1. Notebook'un üstündeki **Restart** düğmesine basın.
2. Aşağıdaki Bölüm 3'ün **ilk kod hücresine** gelin → hücrenin `...` menüsü → **Execute Above Cells**.

## 3. Worker agent: konuya göre araç seti

Bölüm 1'de araç listesini elle seçtik. Gerçekte talep kartla da, hesapla da, krediyle de ilgili olabilir.
Her seferinde bütün araçları vermek yerine iki adım izliyoruz:

1. Önce talebin **konusunu** buluyoruz. Bunu `siniflandirici.py` yapıyor: tek model çağrısı ve
   yapılandırılmış çıktı (Hafta 1'de gördüğünüz teknik).
2. Ajana sadece **o konunun araçlarını** veriyoruz. Hangi konunun hangi araçları kullanacağı
   `src/asistan/kategoriler.py` dosyasında yazıyor.

Ajan **tek**: aynı `ajan_calistir` fonksiyonu. Değişen sadece elindeki alet çantası.
Belirli bir iş için sınırlı araçlarla çalışan bu ajana **worker agent** (işçi ajan) diyoruz.

Neden? (1) Araç az olunca model daha az karışır. (2) Kart talebiyle uğraşan ajan hesap verisine hiç dokunamaz.

In [ ]:
from asistan.kategoriler import KATEGORILER

for kategori, tanim in KATEGORILER.items():
    print(f"{kategori:<6} → {tanim['araclar']}")

In [ ]:
from asistan.araclar import OKUMA_ARACLARI
from asistan.siniflandirici import siniflandir


def arac_seti(kategori):
    """kategoriler.py'deki araç adlarını gerçek araçlara çevirir."""
    adlar = KATEGORILER[kategori]["araclar"]
    return [arac for arac in OKUMA_ARACLARI if arac.name in adlar]


talep = "Hesabımda neden bloke var?"
kategori = siniflandir(talep, MODEL).kategori
print("Kategori:", kategori, "| araçlar:", [arac.name for arac in arac_seti(kategori)], "\n")

print("\nBulgular:\n" + ajan_calistir(talep, "M0012", arac_seti(kategori)))

**Deneyin:** Aynı hücreyi `TALEP` ve `MUSTERI_NO` ile çalıştırın. Kategori ve araç seti nasıl değişti?

Not: `guvenlik` (kötü niyetli talep) ve `diger` (bankacılık dışı) kategorileri ajana hiç gitmez,
hazır bir yanıt döner. Bunu Bölüm 6'da ekleyeceğiz.

## 4. Güvenlik sınırları ve insan onayı

Şu ana kadar ajan sadece **okudu**. Müşteri ise iade istiyor. İade bir **yazma** işlemidir, yani para hareket ettirir.

Kural: **yazma aracı modele hiç verilmez.** `araclar.py` içinde araçlar iki ayrı listede duruyor:

In [ ]:
from asistan.araclar import YAZMA_ARACLARI

print("Modele verilenler   :", [arac.name for arac in OKUMA_ARACLARI])
print("Modele VERİLMEYENLER:", [arac.name for arac in YAZMA_ARACLARI])

Peki iade nasıl yapılacak? Üç adımda:

1. Ajan veriyi okur ve bulgularını yazar (Bölüm 1).
2. Bulgulardan bir **yanıt taslağı** çıkarılır. İade gerekiyorsa taslakta bir **iade önerisi** de olur.
   Bu yapılandırılmış çıktıdır: model sadece öneri yazar, işlem yapamaz.
3. Bir **insan** öneriyi onaylarsa iade aracını **bizim kodumuz** çağırır. Reddederse talep temsilciye gider.

In [ ]:
from pydantic import BaseModel, Field


class Taslak(BaseModel):
    yanit: str = Field(description="Müşteriye gidecek yanıt: Türkçe, nazik, en fazla 4 cümle.")
    iade_islem_id: str = Field(default="", description="İade edilecek işlemin numarası, bulgulardan AYNEN "
                                                       "kopyalanır (ör. I00123). İade gerekmiyorsa boş.")
    iade_tutari: float = Field(default=0, description="İade edilecek işlemin tutarı (TL), bulgulardan "
                                                      "kopyalanır. İade gerekmiyorsa 0.")


TASLAK_SISTEM = ("Sen Örnek Bank'ın müşteri asistanısın. Ajanın bulgularına bakarak müşteriye yanıt taslağı yaz.\n"
                 "Kurallar:\n"
                 "- Sadece bulgulardaki bilgiyi kullan; numara, tutar veya tarih uydurma.\n"
                 "- Bulgularda aynı gün aynı tutarda iki işlem varsa (mükerrer çekim) iade öner: iade_islem_id "
                 "alanına İKİNCİ işlemin numarasını, iade_tutari alanına o işlemin tutarını bulgulardan AYNEN "
                 "kopyala. Prosedür bilgisi olmasa da bunu yap.\n"
                 "- İadeyi sen yapamazsın. Yanıtta 'iade talebiniz onaya iletildi' de; 'iade edildi', "
                 "'onaylıyorum' gibi ifadeler kullanma.")


def taslak_yaz(talep, bulgular):
    yontem = "function_calling" if MODEL == "yerel" else "json_schema"   # iki model farklı yöntemi destekliyor
    model = model_getir(MODEL).with_structured_output(Taslak, method=yontem)
    return model.invoke([SystemMessage(TASLAK_SISTEM),
                         HumanMessage(f"Müşteri talebi: {talep}\n\nAjanın bulguları:\n{bulgular}")])


taslak = taslak_yaz(TALEP, bulgular)      # bulgular: Bölüm 1'de M0007 için topladığımız
print(taslak)

In [ ]:
from asistan.araclar import ApiHatasi, iade_talebi_olustur


def onay_gerekli_mi(taslak):
    """İade önerisi varsa YA DA yanıtta 'iade' geçiyorsa bir insan görmeli (proje kuralı)."""
    yanit = taslak.yanit.replace("İ", "i").lower()   # Python'da "İ".lower() Türkçe "i" vermez, önce düzeltiyoruz
    return bool(taslak.iade_islem_id) or "iade" in yanit


def insan_onayi(taslak, musteri_no):
    print(f"YANIT TASLAĞI: {taslak.yanit}")
    if taslak.iade_islem_id:
        print(f"İADE ÖNERİSİ : {taslak.iade_islem_id} numaralı işlem için {taslak.iade_tutari} TL")
    karar = input("Onaylıyor musunuz? (e/h): ")
    if karar.strip().lower() != "e":
        return "TEMSİLCİYE AKTAR: temsilci onaylamadı"
    if not taslak.iade_islem_id:             # onaylandı ama açılacak bir iade yok: sadece yanıt gider
        return "onaylandı"

    try:
        with musteri_oturumu(musteri_no):    # yazma aracını MODEL değil, onaydan sonra KOD çağırıyor
            return iade_talebi_olustur.invoke({"islem_id": taslak.iade_islem_id, "tutar": taslak.iade_tutari,
                                               "sebep": "mükerrer aidat", "onaylayan": "egitim_temsilcisi"})
    except ApiHatasi as hata:                # servis de kontrol ediyor: işlem bu müşterinin değilse reddeder
        return f"TEMSİLCİYE AKTAR: iade açılamadı ({hata})"


if onay_gerekli_mi(taslak):
    print(insan_onayi(taslak, MUSTERI_NO))
else:
    print("Onay gerekmiyor, yanıt doğrudan gider:", taslak.yanit)

**Deneyin:** Hücreyi bir kez `e`, bir kez `h` diyerek çalıştırın.
Düşünün: Model iade aracını kendisi çağırmak isteseydi ne olurdu? (İpucu: elinde böyle bir araç yok.)

**Modele güvenmiyoruz, kontrol ediyoruz.** Model hata yapabilir. Üç ayrı kontrol var:
- `onay_gerekli_mi` sadece öneri alanına bakmıyor, yanıtta "iade" kelimesi geçiyor mu diye de bakıyor.
  Model alanı boş bırakıp yanıtta iade sözü verirse yanıt yine de insana gider.
  (Küçük tuzak: Python'da `"İade".lower()` → `"i̇ade"` olur ve `"iade"` aramasını kaçırır. Bu yüzden önce `İ`'yi düzeltiyoruz.)
- **İnsan** taslağı ve öneriyi görüp reddedebilir: yanlış numara, eksik tutar...
- **Servis** işlemin bu müşteriye ait olup olmadığını ve tutarı ayrıca kontrol eder.

Bugün gördüğümüz güvenlik sınırları:

| Sınır | Nerede |
|---|---|
| Müşteri numarası modelden gelmez | `araclar.py` → `aktif_musteri()` |
| Araç verisi talimat sayılmaz | `veri_olarak_etiketle` (`<arac_verisi>` etiketi) |
| Yazma aracı modele verilmez | `OKUMA_ARACLARI` / `YAZMA_ARACLARI` |
| Para hareketi insan onayı ister | bu bölüm (`ajan.py` → `insan_onayi_dugumu`) |
| Döngü sonsuz olamaz | adım limiti (Bölüm 1) |

## 5. Hata yönetimi ve fallback

Gerçek servisler bazen yanıt vermez. Mock API de bilerek ara sıra `503` hatası döner (`.env` → `MOCK_HATA_ORANI`).
Kuralımız iki adımlı:

1. **Yeniden dene:** Hata geçici olabilir, 2 kez daha dene. Bunu araçların içindeki `api_cagir` zaten yapıyor.
2. **Fallback (yedek yol):** Yine olmazsa cevap uydurma, talebi **temsilciye aktar**. Buna "güvenli taraf" diyoruz.

Servisi "çökertelim": adresi bilerek boş bir porta çeviriyoruz.

In [ ]:
import os
from asistan.araclar import ApiHatasi   # yeniden denemelere rağmen servis yanıt vermezse gelen hata

DOGRU_ADRES = os.environ["MOCK_API_URL"]
os.environ["MOCK_API_URL"] = "http://localhost:9999"      # bu adreste servis yok

try:
    with musteri_oturumu(MUSTERI_NO):
        kart_islemleri.invoke({})
except ApiHatasi as hata:
    print("Yakalanan hata:", hata)
finally:
    os.environ["MOCK_API_URL"] = DOGRU_ADRES             # adresi geri al

Uyarı satırlarında 3 deneme görünüyor: ilk deneme + 2 yeniden deneme. Sonunda `ApiHatasi` geldi.

Şimdi bu hatayı ajan döngüsünde yakalayalım. `ajan_calistir` fonksiyonuna iki şey ekliyoruz (`# YENİ` satırları):
- Araç hata verirse döngüden çık ve **temsilciye aktar**.
- Model listede olmayan bir araç isterse onu çalıştırma, modele "böyle bir araç yok" de.

In [ ]:
def ajan_calistir(talep, musteri_no, araclar, adim_limiti=6):
    model = model_getir(MODEL).bind_tools(araclar)
    arac_bul = {arac.name: arac for arac in araclar}
    mesajlar = [SystemMessage(SISTEM), HumanMessage(talep)]

    for adim in range(1, adim_limiti + 1):
        cevap = model.invoke(mesajlar)
        mesajlar.append(cevap)

        if not cevap.tool_calls:
            print(f"adım {adim}: araç yok, bulgular yazıldı")
            return cevap.text

        for cagri in cevap.tool_calls:
            print(f"adım {adim}: {cagri['name']}({cagri['args']})")
            if cagri["name"] not in arac_bul:                                   # YENİ: uydurma araç
                sonuc = f"HATA: '{cagri['name']}' adında bir araç yok. Kullanabileceklerin: {list(arac_bul)}"
            else:
                try:
                    with musteri_oturumu(musteri_no):
                        sonuc = arac_bul[cagri["name"]].invoke(cagri["args"])
                except ApiHatasi as hata:                                       # YENİ: servis hatası
                    return f"TEMSİLCİYE AKTAR: araç hatası ({hata})"
            mesajlar.append(ToolMessage(sonuc, tool_call_id=cagri["id"]))

    return "TEMSİLCİYE AKTAR: adım limiti aşıldı"

In [ ]:
os.environ["MOCK_API_URL"] = "http://localhost:9999"      # servisi yine "çökertiyoruz"
try:
    print(ajan_calistir(TALEP, MUSTERI_NO, arac_seti("kart")))
finally:
    os.environ["MOCK_API_URL"] = DOGRU_ADRES

Ajan cevap uydurmadı, talebi temsilciye bıraktı. Bankada doğru davranış bu: **emin değilsen insana bırak.**

## 6. Hepsi bir arada

Şimdi bütün parçaları tek bir fonksiyonda birleştiriyoruz. Her satır, önceki bölümlerden birinde yazdığımız bir parça:

In [ ]:
from asistan.kategoriler import AJANA_GITMEYENLER


def talep_isle(talep, musteri_no):
    # 1) Konu ne? (Bölüm 3)
    kategori = siniflandir(talep, MODEL).kategori
    print("kategori:", kategori)

    # 2) Kötü niyetli ya da bankacılık dışı talep: ajana gitmez, hazır yanıt (Bölüm 3)
    if kategori in AJANA_GITMEYENLER:
        return AJANA_GITMEYENLER[kategori]["hazir_yanit"]

    # 3) Worker ajan: sadece bu konunun araçlarıyla bilgi toplar (Bölüm 1, 3, 5)
    bulgular = ajan_calistir(talep, musteri_no, arac_seti(kategori))
    if bulgular.startswith("TEMSİLCİYE AKTAR"):
        return bulgular

    # 4) Yanıt taslağı; iade varsa insan onayı (Bölüm 4)
    taslak = taslak_yaz(talep, bulgular)
    if onay_gerekli_mi(taslak):
        sonuc = insan_onayi(taslak, musteri_no)
        if sonuc.startswith("TEMSİLCİYE AKTAR"):
            return sonuc

    return taslak.yanit

In [ ]:
for talep, musteri_no in [
    (TALEP, MUSTERI_NO),                                  # kart + iade → insan onayı sorar
    ("Hesabımda neden bloke var?", "M0012"),              # hesap → Lab 3'teki aracınız
    ("Bana bir aşk şiiri yazar mısın?", "M0001"),         # diger → hazır yanıt, araç yok
]:
    print(f"\n=== {musteri_no}: {talep}")
    print("YANIT:", talep_isle(talep, musteri_no))

## Yazdığınız her parça `ajan.py`'de nerede?

Bu notebook'ta ajanı ~60 satırda kurduk. `src/asistan/ajan.py` aynı akışın projedeki hali:

| Bu notebook | `src/asistan/ajan.py` |
|---|---|
| `siniflandir(...)`, `AJANA_GITMEYENLER` | `siniflandir_dugumu`, `hazir_yanit_dugumu` |
| `arac_seti(kategori)` | `arac_setini_sec` |
| `ajan_calistir` döngüsü | `ajan_dugumu` ⇄ `arac_calistir_dugumu` |
| `adim_limiti=6` | `ADIM_LIMITI` |
| `except ApiHatasi` → temsilciye aktar | `temsilciye_aktar_dugumu` |
| `taslak_yaz`, `Taslak` | `taslak_dugumu`, `Taslak` |
| `onay_gerekli_mi` | `risk_kontrolu_dugumu` |
| `insan_onayi` | `insan_onayi_dugumu` |

`ajan.py` bunlara ek olarak şunları yapıyor: talep belirsizse
netleştirme sorusu soruyor, risk kontrolünde iadenin yanında tutar ve hukuki konuya da bakıyor ve her adımı logluyor.
Parçaları **LangGraph** ile bağlıyor: her parça bir *düğüm* (fonksiyon), aralarındaki oklar `graf_olustur()`
içinde. Mantık aynı. LangGraph sadece "sıradaki adım hangisi" kısmını düzenli tutuyor ve insan onayında
akışın bekleyebilmesini sağlıyor.

Terminalde aynı talebi projenin ajanıyla çalıştırın:
```bash
python -m asistan.ajan "Kartımdan yıllık aidat iki kez kesilmiş, fazla olanın iadesini istiyorum." M0007
```
Çıktıdaki her satırın bu notebook'ta hangi parçaya karşılık geldiğini bulun.

## Tartışma soruları
1. Adım limiti olmasaydı, Bölüm 1'deki döngü hangi durumda hiç bitmezdi?
2. Kart talebiyle uğraşan ajana hesap araçlarını vermemek neyi korur?
3. İade önerisini model yazıyor, işlemi kod yapıyor. Bu ayrım neden önemli?
4. Servis çöktüğünde ajan "temsilciye aktar" demek yerine kendi bildiğini söyleseydi ne olurdu?